In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
from sklearn import set_config
set_config(display='text')

In [3]:
csv_path = "./dataset/diamonds.csv"
print(f"시작: {csv_path} 파일을 로드")
df = pd.read_csv(csv_path)

# 불필요한 데이터 열 삭제
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

시작: ./dataset/diamonds.csv 파일을 로드


FileNotFoundError: [Errno 2] No such file or directory: './dataset/diamonds.csv'

In [ ]:
print("범주형 변수 수동 원-핫 인코딩")

# 다중공선성 방지를 위해 drop_first=True 설정
# 문자열 컬럼들이 0과 1의 수치형 컬럼들로 분리됩니다.
df_encoded = pd.get_dummies(df, columns=['cut', 'color', 'clarity'], drop_first=True, dtype=int)

# 피처(X)와 타겟(y) 분리
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

# 학습 및 검증 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 원본 데이터프레임 백업 (나중에 다항 변환용 데이터로 활용)
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

In [ ]:
print("수치형 변수 수동 스케일링")
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']

scaler = StandardScaler()

# 훈련 데이터의 수치형 컬럼들만 표준화 기준 학습 및 변환
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# 테스트 데이터는 훈련 데이터 기준으로 변환만 수행
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

In [ ]:
print("\n[모델 1] 기본 선형 회귀 학습")
start_time = time.time()

lr_basic = LinearRegression()
lr_basic.fit(X_train, y_train)
y_pred_basic = lr_basic.predict(X_test)

time_basic = time.time() - start_time

rmse_basic = np.sqrt(mean_squared_error(y_test, y_pred_basic))
r2_basic = r2_score(y_test, y_pred_basic)

In [ ]:
print("수동 다항 피처 생성")
start_time = time.time()

# degree=2 설정으로 제곱항 및 교차항 생성 변환기 정의
poly = PolynomialFeatures(degree=2, include_bias=False)

# 전처리가 완료된 X_train과 X_test 행렬 전체를 다항식으로 확장합니다.
X_train_poly = poly.fit_transform(X_train_raw)
X_test_poly = poly.transform(X_test_raw)

# 변환된 다항 데이터를 가지고 새 선형 회귀 모델 학습
lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly = lr_poly.predict(X_test_poly)

time_poly = time.time() - start_time
print(time_poly)

rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly = r2_score(y_test, y_pred_poly)


In [ ]:
print("\n" + "="*65)
print("     [일반 선형 회귀 vs 다항 회귀 비교]")
print("="*65)

results = [
    {
        "Model": "기본",
        "Features 수": X_train.shape[1],
        "RMSE": round(rmse_basic, 2),
        "R2 Score": round(r2_basic, 4),
        "Time (초)": round(time_basic, 2)
    },
    {
        "Model": "다항",
        "Features 수": X_train_poly.shape[1],
        "RMSE": round(rmse_poly, 2),
        "R2 Score": round(r2_poly, 4),
        "Time (초)": round(time_poly, 2)
    }
]

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print("="*65)

$6\text{(수치형)} + 4\text{(cut)} + 6\text{(color)} + 7\text{(clarity)} = \mathbf{23\text{개}}$
<br>
 $23\text{(본래 항)} + 23\text{(제곱 항)} + 253\text{(교차 항)} = \mathbf{299\text{개}}$

<hr>

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
csv_path = "./dataset/diamonds.csv"

print(f"시작: {csv_path} 파일을 로드")
df = pd.read_csv(csv_path)
print(f"데이터 로드 완료! 데이터 형태: {df.shape}")

In [ ]:
# 캐글 등에서 받은 csv에 흔히 포함된 인덱스 컬럼('Unnamed: 0')이 있다면 제거합니다.
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

X = df.drop(columns=['price'])
y = df['price']

# 학습 및 검증 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42)

In [ ]:
# [단계 3] 전처리 파이프라인 구성 (수치형 표준화 & 범주형 원-핫 인코딩)
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']
categorical_features = ['cut', 'color', 'clarity']

# ColumnTransformer 설정
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

print("전처리: 스케일링 및 원-핫 인코딩")
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [ ]:
print("\n[모델 1] 기본 선형 회귀 학습")
start_time = time.time()

lr_basic = LinearRegression()
lr_basic.fit(X_train_preprocessed, y_train)
y_pred_basic = lr_basic.predict(X_test_preprocessed)

time_basic = time.time() - start_time

# 평가지표 계산
rmse_basic = np.sqrt(mean_squared_error(y_test, y_pred_basic))
r2_basic = r2_score(y_test, y_pred_basic)

In [ ]:
print("다항 선형 회귀(Polynomial Regression) 학습")
start_time = time.time()

# degree=2 설정으로 제곱항 및 상호작용항 생성
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('linear', LinearRegression())
])

poly_pipeline.fit(X_train_preprocessed, y_train)
y_pred_poly = poly_pipeline.predict(X_test_preprocessed)

time_poly = time.time() - start_time

# 평가지표 계산
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly = r2_score(y_test, y_pred_poly)

# 생성된 피처 개수 세기
num_features_basic = X_train_preprocessed.shape[1]
num_features_poly = poly_pipeline.named_steps['poly'].transform(X_train_preprocessed).shape[1]

In [ ]:
print("\n" + "="*65)
print("            [일반 선형 회귀 vs 다항 선형 회귀 성능 비교]")
print("="*65)

results = [
    {
        "Model": "기본 회귀",
        "Features 수": num_features_basic,
        "RMSE": round(rmse_basic, 2),
        "R2 Score": round(r2_basic, 4),
        "Time (초)": round(time_basic, 2)
    },
    {
        "Model": "다항 회귀",
        "Features 수": num_features_poly,
        "RMSE": round(rmse_poly, 2),
        "R2 Score": round(r2_poly, 4),
        "Time (초)": round(time_poly, 2)
    }
]

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print("="*65)

<hr>

#### 회귀모델 비교
최근 머신러닝 트렌드에서 가장 주목받는 LightGBM, CatBoost, XGBoost, Random Forest, ElasticNet 비교

In [ ]:
#!pip install xgboost lightgbm catboost pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# [전통적 모델]
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor

# [최신 트렌드 모델 (Modern Gradient Boosting)]
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
csv_path = "./dataset/diamonds.csv"
print(f"시작: {csv_path} 파일을 로드")
df = pd.read_csv(csv_path)

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

X = df.drop(columns=['price'])
y = df['price']

# 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']
categorical_features = ['cut', 'color', 'clarity']

# 최신 트리 모델들은 다중공선성에 유연하므로 원-핫 인코딩 시 drop='first'를 빼서
# 모든 범주 정보를 온전히 보존하는 것이 성능에 유리합니다.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

print("전처리: 스케일링 및 원-핫 인코딩 진행")
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

In [ ]:
models = {
    # 1. ElasticNet: L1(Lasso)과 L2(Ridge) 규제를 결합한 전통 선형 회귀의 최종 진화형
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),

    # 2. Random Forest: 배깅(Bagging) 방식, 벤치마크 기준점
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),

    # 3. XGBoost: 병렬 연산과 시스템 최적화
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),

    # 4. LightGBM: 대용량 데이터에서 압도적인 속도와 적은 메모리를 자랑하는 리프 중심(Leaf-wise) 트리 모델
    "LightGBM": LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1),

    # 5. CatBoost: 범주형 변수(Categorical) 처리에 특화되어 과적합을 방지하는 최근 가장 핫한 모델
    "CatBoost": CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
}

In [ ]:
results = []

print("\n=== 대규모 회귀 알고리즘 배틀 시작 ===")
for name, model in models.items():
    start_time = time.time()

    # 모델 학습
    model.fit(X_train_scaled, y_train)

    # 예측
    y_pred = model.predict(X_test_scaled)

    # 시간 측정
    elapsed_time = time.time() - start_time

    # 성능 지표 계산
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "RMSE": round(rmse, 2),
        "R2 Score": round(r2, 4),
        "Training Time (초)": round(elapsed_time, 2)
    })
    print(f"[{name}] 완료! (소요 시간: {elapsed_time:.2f}초)")

In [ ]:
print("\n" + "="*70)
print("             [최신 트렌드 반영 회귀 알고리즘 성능 비교 표]")
print("="*70)

df_results = pd.DataFrame(results)
# R2 Score 기준으로 내림차순 정렬 (성능이 가장 좋은 모델이 맨 위로)
df_results = df_results.sort_values(by="R2 Score", ascending=False).reset_index(drop=True)

print(df_results.to_string(index=False))
print("="*70)

In [ ]:
df_results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="white")
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 5))

# 성능이 높은 모델일수록 진한 파란색이 되도록  palette 설정
sns.barplot(
    x="R2 Score",
    y="Model",
    data=df_results,
    palette="Blues",
    hue="Model",
    legend=False
)

# 4. 가독성을 위한 그래프 디테일 조정
plt.title("회귀 모델별 설명력 ($R^2$ Score) 비교", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("예측 정확도 (1에 가까울수록 완벽)", fontsize=11)
plt.ylabel("") # 불필요한 Y축 라벨 생략
plt.xlim(0.8, 1.0) # 차이를 명확히 보기 위해 0.8~1.0 구간만 확대

# 막대 끝에 정확한 수치 표시
for p in plt.gca().patches:
    width = p.get_width()
    plt.gca().text(
        width + 0.002,
        p.get_y() + p.get_height()/2,
        f'{width*100:.1f}%', # 직관적인 이해를 위해 퍼센트(%) 형태로 표기
        ha="left", va="center", fontsize=10, fontweight='bold', color='#333333'
    )

# 테두리 선 제거로 더 깔끔하게 처리
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()